In [0]:
dbutils.widgets.text("catalog","sl")
catalog = dbutils.widgets.get("catalog")

## 1. Overall KPI Summary — Net Revenue, Distinct Bookings, Active Passengers, Average Booking Value

All-time snapshot of core commercial performance across the full booking history, built on the gold KPI view `sl.gold.vw_annual_booking_kpi`.

In [0]:
display(spark.sql(f"""
WITH gold_annual AS (
 
    SELECT
        SUM(annual_revenue)   AS net_revenue,
        SUM(distinct_bookings) AS distinct_bookings
    FROM {catalog}.gold.vw_annual_booking_kpi
),
overall_passengers AS (
    SELECT COUNT(DISTINCT bk_passengerid) AS active_passengers
    FROM {catalog}.silver.booking
)
SELECT
    ROUND(net_revenue, 2)                     AS net_revenue,
    distinct_bookings,
    active_passengers,
    ROUND(net_revenue / distinct_bookings, 2)  AS avg_booking_value
FROM gold_annual CROSS JOIN overall_passengers
"""))

### Interpretation & Management Action

The business generated **$60.46M in net revenue** from **8,000 distinct bookings** made by **1,494 active passengers**, at an average booking value of **$7,557.92** — a premium, long-haul-oriented customer base given the per-booking spend. With only ~1,494 passengers driving $60M of revenue (~$40.4K lifetime value per active passenger), the business is highly dependent on a small frequent-flyer base.

**Suggested management action:** Prioritize the loyalty program (targeted retention offers, tier upgrades) to protect the small high-value passenger base, and launch acquisition campaigns to broaden the active passenger pool, since each retained passenger is worth ~$40K in lifetime bookings.

## 2. Monthly Revenue Trend Across All Years

Chronological monthly net revenue with month-over-month growth, from the booking base behind the gold KPI view `sl.gold.vw_annual_booking_kpi`.

In [0]:
display(spark.sql(f"""
WITH monthly AS (
    SELECT
        DATE_TRUNC('month', bk_bookingdate) AS month,
        SUM(bk_totalprice)                   AS monthly_revenue,
        COUNT(*)                             AS bookings
    FROM {catalog}.silver.booking
    WHERE bk_bookingdate IS NOT NULL
    GROUP BY DATE_TRUNC('month', bk_bookingdate)
)
SELECT
    month,
    year(month)                AS year,
    date_format(month, 'MMMM') AS month_name,
    ROUND(monthly_revenue, 2)                                         AS monthly_revenue,
    bookings,
    ROUND(100 * (monthly_revenue - LAG(monthly_revenue) OVER w)
          / LAG(monthly_revenue) OVER w, 1)                            AS mom_growth_pct
FROM monthly
WINDOW w AS (ORDER BY month)
ORDER BY month
"""))

Databricks visualization. Run in Databricks to view.

### Interpretation & Management Action

Monthly revenue is remarkably **stable, oscillating in a ~$0.8M–$1.2M band around a ~$1.0M monthly baseline** across the full 5-year history (1994–1998), with no sustained growth or decline. Peaks cluster in **January** and **July/August** (holiday travel) while the deepest troughs are **December 1995 ($803K)** and **February ($839K–$999K)**, and month-over-month swings of ±15–30% point to seasonal demand patterns rather than a structural trend.

**Suggested management action:** Treat ~$1.0M as the monthly revenue baseline with ±20% seasonal bands for target-setting, run counter-seasonal promotions (dynamic discounts, package deals) in trough months like December and February, and protect capacity/pricing in the January and summer peaks — the business is defending a stable base, not riding growth.

## 3. Annual Revenue with Year-over-Year Change

Yearly net revenue, absolute YoY change, and YoY growth rate, built directly on the gold KPI view `sl.gold.vw_annual_booking_kpi`.

In [0]:
display(spark.sql(f"""
WITH annual AS (
    SELECT
        booking_year,
        annual_revenue,
        distinct_bookings
    FROM {catalog}.gold.vw_annual_booking_kpi
)
SELECT
    booking_year                                        AS year,
    ROUND(annual_revenue, 2)                           AS annual_revenue,
    ROUND(annual_revenue
          - LAG(annual_revenue) OVER w, 2)             AS yoy_change_abs,
    ROUND(100 * (annual_revenue - LAG(annual_revenue) OVER w)
          / LAG(annual_revenue) OVER w, 1)               AS yoy_change_pct,
    distinct_bookings
FROM annual
WINDOW w AS (ORDER BY booking_year)
ORDER BY booking_year
"""))

### Interpretation & Management Action

Annual revenue is essentially **flat, ranging $11.4M–$12.5M** with small oscillations: a **-6.8% dip in 1995** ($11.43M) recovered by **+6.3% in 1996**, followed by a near-flat 1997 (-0.5%) and the **best year in 1998 ($12.54M, +3.7%)**. No year shows structural growth — the ±7% swings track booking volume (1,519–1,646 bookings), not pricing, so revenue moves are volume-driven.

**Suggested management action:** Investigate what drove the 1995 volume drop (capacity cuts, pricing, competition) to avoid a repeat, and replicate the 1998 playbook — growing bookings by ~3% while holding the ~$7.6K average booking value is the realistic near-term growth lever, since revenue follows volume in this business.

## 4. Revenue by Passenger Travel Purpose

Revenue, bookings, and passengers segmented by travel purpose, joining the booking base behind the gold KPI view to the gold dimension `sl.gold.dim_passenger`.

In [0]:
display(spark.sql(f"""
WITH purpose_revenue AS (
    SELECT
        p.travel_purpose,
        COUNT(DISTINCT b.bk_bookingid)   AS distinct_bookings,
        COUNT(DISTINCT b.bk_passengerid) AS passengers,
        SUM(b.bk_totalprice)             AS revenue
    FROM {catalog}.silver.booking b
    JOIN {catalog}.gold.dim_passenger p
      ON b.bk_passengerid = p.passenger_id
    GROUP BY p.travel_purpose
)
SELECT
    travel_purpose,
    ROUND(revenue, 2)                          AS revenue,
    ROUND(100 * revenue / SUM(revenue) OVER (), 1) AS revenue_share_pct,
    distinct_bookings,
    passengers,
    ROUND(revenue / distinct_bookings, 2)      AS avg_booking_value,
    ROUND(revenue / passengers, 2)             AS revenue_per_passenger
FROM purpose_revenue
ORDER BY revenue DESC
"""))

Databricks visualization. Run in Databricks to view.

### Interpretation & Management Action

Revenue is **almost perfectly balanced across all five travel purposes** — BUSINESS leads with $12.77M (21.1%), followed by LEISURE ($12.23M), MILITARY ($11.84M), CREW ($11.81M), and STUDENT ($11.81M), each holding 19.5–21.1% share. No segment dominates; average booking values are also tightly clustered ($7.4K–$7.7K), with BUSINESS and MILITARY slightly premium and LEISURE lowest, while LEISURE delivers the highest lifetime revenue per passenger (~$42.0K).

**Suggested management action:** This balanced mix is a strength — no single segment's downturn can sink revenue — so run purpose-tailored programs (corporate negotiated rates for BUSINESS, student/military fares to defend share, bundled leisure packages) rather than betting on one segment, and cross-sell aggressively since multi-purpose frequent flyers drive the ~$40K per-passenger lifetime value.

## 5. Top 10 Passengers by Net Revenue

Highest-value individual customers by lifetime net revenue, joining the booking base behind the gold KPI view to the gold dimension `sl.gold.dim_passenger`.

In [0]:
display(spark.sql(f"""
WITH passenger_revenue AS (
    SELECT
        p.passenger_id,
        p.passenger_name,
        p.travel_purpose,
        p.loyalty_points,
        COUNT(DISTINCT b.bk_bookingid) AS bookings,
        SUM(b.bk_totalprice)           AS net_revenue
    FROM {catalog}.silver.booking b
    JOIN {catalog}.gold.dim_passenger p
      ON b.bk_passengerid = p.passenger_id
    GROUP BY p.passenger_id, p.passenger_name, p.travel_purpose, p.loyalty_points
),
ranked AS (
    SELECT
        passenger_id,
        passenger_name,
        travel_purpose,
        loyalty_points,
        bookings,
        net_revenue,
        ROW_NUMBER() OVER (ORDER BY net_revenue DESC)              AS rank,
        ROUND(100 * net_revenue / SUM(net_revenue) OVER (), 2)     AS revenue_share_pct
    FROM passenger_revenue
)
SELECT
    rank,
    passenger_name,
    travel_purpose,
    loyalty_points,
    bookings,
    ROUND(net_revenue, 2)   AS net_revenue,
    revenue_share_pct
FROM ranked
WHERE rank <= 10
ORDER BY rank
"""))

Databricks visualization. Run in Databricks to view.

### Interpretation & Management Action

The top 10 passengers each generated **$100K–$125K in lifetime net revenue** (10–15 bookings), together contributing **~1.9% of total revenue** — a healthy but not dangerously concentrated tail of high-value frequent flyers. The list spans all five travel purposes, confirming value is spread across segments rather than one customer type; notably, **rank #3 Mary Williams has just 203 loyalty points against $118K in spend**, showing loyalty points do not currently reflect actual revenue contribution.

**Suggested management action:** Enroll these 10 passengers in a VIP/white-glove retention program (dedicated support, guaranteed upgrades) and audit the loyalty-points model — rewarding points in proportion to revenue contribution would make the program a true retention lever for the highest-spending flyers.

## 6. Top 10 Passengers as a Share of Total Revenue

Revenue concentration of the 10 highest-value passengers versus the remaining base, joining the booking base behind the gold KPI view to the gold dimension `sl.gold.dim_passenger`.

In [0]:
display(spark.sql(f"""
WITH passenger_revenue AS (
    SELECT
        p.passenger_id,
        SUM(b.bk_totalprice) AS net_revenue
    FROM {catalog}.silver.booking b
    JOIN {catalog}.gold.dim_passenger p
      ON b.bk_passengerid = p.passenger_id
    GROUP BY p.passenger_id
),
ranked AS (
    SELECT
        net_revenue,
        ROW_NUMBER() OVER (ORDER BY net_revenue DESC) AS rank,
        COUNT(*)   OVER ()                        AS total_passengers,
        SUM(net_revenue) OVER ()                 AS total_revenue
    FROM passenger_revenue
)
SELECT
    total_passengers,
    ROUND(total_revenue, 2)                                         AS total_revenue,
    ROUND(SUM(CASE WHEN rank <= 10 THEN net_revenue END), 2)         AS top_10_revenue,
    ROUND(100 * SUM(CASE WHEN rank <= 10 THEN net_revenue END)
          / total_revenue, 2)                                        AS top_10_share_pct,
    ROUND(AVG(CASE WHEN rank <= 10 THEN net_revenue END), 2)         AS top_10_avg_revenue,
    total_passengers - 10                                           AS remaining_passengers,
    ROUND(100 - 100 * SUM(CASE WHEN rank <= 10 THEN net_revenue END)
          / total_revenue, 2)                                        AS remaining_share_pct,
    ROUND(SUM(CASE WHEN rank > 10 THEN net_revenue END)
          / (total_passengers - 10), 2)                              AS remaining_avg_revenue,
    ROUND(AVG(CASE WHEN rank <= 10 THEN net_revenue END)
          / (SUM(CASE WHEN rank > 10 THEN net_revenue END)
             / (total_passengers - 10)), 1)                          AS top10_vs_rest_multiplier
FROM ranked
GROUP BY total_passengers, total_revenue
"""))

### Interpretation & Management Action

The top 10 passengers contribute **$1.11M — only 1.83% of the $60.46M total** — from 0.7% of the 1,494 active passengers, while the remaining 98.17% of revenue is spread across 1,484 flyers. Each top-10 passenger averages **$110.5K lifetime revenue, 2.8× the ~$40K average** of the rest — valuable, but the business is **not dangerously dependent** on a few customers.

**Suggested management action:** Concentration risk is low, so growth should come from the middle of the base: build tiered loyalty benefits and upsell programs that move the ~1,484 average flyers toward top-10 behavior (2–3× more bookings each), while still assigning key-account managers to the top 10 as a defensive measure.

## 7. Top 10 Routes by Net Revenue

Highest-revenue routes by discounted line-item revenue (ticket price × (1 − discount) × seats, refunded segments excluded), joining the flight_booking fact to the gold dimension `sl.gold.dim_route`.

In [0]:
display(spark.sql(f"""
WITH route_revenue AS (
    SELECT
        r.route_id,
        r.route_name,
        r.route_type,
        r.airline_brand,
        COUNT(DISTINCT fb_bookingid)                       AS bookings,
        SUM(fb_seats)                                      AS seats_sold,
        SUM(fb_ticketprice * (1 - fb_discount) * fb_seats) AS net_revenue
    FROM {catalog}.silver.flight_booking fb
    JOIN {catalog}.gold.dim_route r
      ON fb.fb_routeid = r.route_id
    WHERE fb_refundflag <> 'R'          -- exclude refunded segments
    GROUP BY r.route_id, r.route_name, r.route_type, r.airline_brand
)
SELECT
    ROW_NUMBER() OVER (ORDER BY net_revenue DESC)              AS rank,
    route_id,
    route_name,
    route_type,
    airline_brand,
    bookings,
    seats_sold,
    ROUND(net_revenue, 2)                                      AS net_revenue,
    ROUND(100 * net_revenue / SUM(net_revenue) OVER (), 2)      AS revenue_share_pct,
    ROUND(net_revenue / seats_sold, 2)                         AS revenue_per_seat
FROM route_revenue
ORDER BY rank
LIMIT 10
"""))

Databricks visualization. Run in Databricks to view.

### Interpretation & Management Action

The top 10 routes each generate **$1.05M–$1.16M in net revenue** with only **1.1–1.2% share each** — no single route dominates, and revenue is volume-driven (600–675 seats sold per route) rather than premium-driven (revenue per seat clusters at $1,680–$1,910). Notably, the #1 and #2 routes (**SIN-LIM and GRU-MAD**) are short-haul, and **Arabian Sky operates 5 of the top 10 routes**, while FRA-ZRH is the efficiency leader at $1,910 per seat.

**Suggested management action:** Since no route is a dependency, optimize the network on margin per seat — study FRA-ZRH's pricing/cabin mix and apply it to low-yield routes like HKG-ACC ($1,678/seat), strengthen Arabian Sky's leading route portfolio with dedicated capacity, and treat the flat ~$1.1M-per-route ceiling as evidence that growth must come from adding routes/frequency, not repricing existing ones.

## 8. Bottom 10 Routes by Net Revenue

Lowest-revenue routes by discounted line-item revenue (ticket price × (1 − discount) × seats, refunded segments excluded), joining the flight_booking fact to the gold dimension `sl.gold.dim_route`.

In [0]:
display(spark.sql(f"""
WITH route_revenue AS (
    SELECT
        r.route_id,
        r.route_name,
        r.route_type,
        r.airline_brand,
        COUNT(DISTINCT fb_bookingid)                       AS bookings,
        SUM(fb_seats)                                      AS seats_sold,
        SUM(fb_ticketprice * (1 - fb_discount) * fb_seats) AS net_revenue
    FROM {catalog}.silver.flight_booking fb
    JOIN {catalog}.gold.dim_route r
      ON fb.fb_routeid = r.route_id
    WHERE fb_refundflag <> 'R'          -- exclude refunded segments
    GROUP BY r.route_id, r.route_name, r.route_type, r.airline_brand
)
SELECT
    ROW_NUMBER() OVER (ORDER BY net_revenue ASC)               AS rank,
    route_id,
    route_name,
    route_type,
    airline_brand,
    bookings,
    seats_sold,
    ROUND(net_revenue, 2)                                      AS net_revenue,
    ROUND(100 * net_revenue / SUM(net_revenue) OVER (), 2)      AS revenue_share_pct,
    ROUND(net_revenue / seats_sold, 2)                         AS revenue_per_seat
FROM route_revenue
ORDER BY rank
LIMIT 10
"""))

Databricks visualization. Run in Databricks to view.

### Interpretation & Management Action

The bottom 10 routes generate **$753K–$837K each** — roughly 30% below the top routes — but none is failing: even the weakest (AMS-SCL, route 48) sold **482 seats at $1,562/seat**. Revenue is spread across all four brands, with **AmeriJet operating 3 of the bottom 10 but none of the top 10**, and a notable data quirk: **FRA-ZRH appears at both extremes as two distinct route products** — route 86 earns $1.91M at $1,910/seat (top-10 #4) while route 10 on the same city pair earns $794K at $1,661/seat (bottom-10 #4).

**Suggested management action:** Don't cut the tail — it still sells seats profitably; instead lift its yield by consolidating duplicated city-pairs onto the better-performing route product (e.g., migrate FRA-ZRH route 10's schedule/pricing toward route 86's configuration), run a route-level profitability review of AmeriJet's portfolio, and target the lowest-yield routes (NBO-SCL at $1,519/seat, LAX-BKK at $1,527/seat) with cabin-mix and pricing adjustments.

## 9. Revenue by Route Type

Net revenue (ticket price × (1 − discount) × seats, refunded segments excluded) grouped by route type, joining the flight_booking fact to the gold dimension `sl.gold.dim_route`.

In [0]:
display(spark.sql(f"""
WITH type_revenue AS (
    SELECT
        r.route_type,
        COUNT(DISTINCT r.route_id)                       AS routes,
        COUNT(DISTINCT fb.fb_bookingid)                  AS bookings,
        SUM(fb.fb_seats)                                 AS seats_sold,
        SUM(fb_ticketprice * (1 - fb_discount) * fb_seats) AS net_revenue
    FROM {catalog}.silver.flight_booking fb
    JOIN {catalog}.gold.dim_route r
      ON fb.fb_routeid = r.route_id
    WHERE fb_refundflag <> 'R'          -- exclude refunded segments
    GROUP BY r.route_type
)
SELECT
    route_type,
    routes,
    bookings,
    seats_sold,
    ROUND(net_revenue, 2)                                   AS net_revenue,
    ROUND(100 * net_revenue / SUM(net_revenue) OVER (), 2)  AS revenue_share_pct,
    ROUND(net_revenue / seats_sold, 2)                      AS revenue_per_seat,
    ROUND(net_revenue / routes, 2)                          AS revenue_per_route
FROM type_revenue
ORDER BY net_revenue DESC
"""))

Databricks visualization. Run in Databricks to view.

### Interpretation & Management Action

Revenue is split almost evenly across the four route types — **ULTRA_LONG_HAUL leads with $26.55M (27.9%)**, followed by LONG_HAUL ($25.46M, 26.7%), SHORT_HAUL ($22.12M, 23.2%), and DOMESTIC ($21.13M, 22.2%) — and each type's share simply mirrors its route count (22–28 routes), with uniform economics of **~$0.94–0.96M revenue per route**. Per-seat net yield is virtually flat ($1,689–$1,705) because average ticket prices (~$1,815–$1,834) and discounts (~6.8–7.0%) barely differ by haul length — pricing does not monetize distance at all.

**Suggested management action:** Introduce haul-based pricing tiers so ULTRA_LONG_HAUL captures a real premium over DOMESTIC instead of the current ~$15/seat edge, review whether costlier long-haul operations are being cross-subsidized by uniform fares, and direct marginal network expansion to ULTRA_LONG_HAUL where the (small) yield edge and route count already lead.

## 10. Top 10 Airlines by Net Revenue

Airline-level net revenue (ticket price × (1 − discount) × seats, refunded segments excluded), joining the flight_booking fact to the gold dimension `sl.gold.dim_airline`. Note: the network operates exactly 10 airlines, so this ranks the full airline universe.

In [0]:
display(spark.sql(f"""
WITH airline_revenue AS (
    SELECT
        a.airline_id,
        a.airline_name,
        COUNT(DISTINCT fb.fb_routeid)                  AS routes,
        COUNT(DISTINCT fb.fb_bookingid)               AS bookings,
        SUM(fb.fb_seats)                               AS seats_sold,
        SUM(fb_ticketprice * (1 - fb_discount) * fb_seats) AS net_revenue
    FROM {catalog}.silver.flight_booking fb
    JOIN {catalog}.gold.dim_airline a
      ON fb.fb_airlineid = a.airline_id
    WHERE fb_refundflag <> 'R'          -- exclude refunded segments
    GROUP BY a.airline_id, a.airline_name
)
SELECT
    ROW_NUMBER() OVER (ORDER BY net_revenue DESC)               AS rank,
    airline_name,
    routes,
    bookings,
    seats_sold,
    ROUND(net_revenue, 2)                                      AS net_revenue,
    ROUND(100 * net_revenue / SUM(net_revenue) OVER (), 2)      AS revenue_share_pct,
    ROUND(net_revenue / seats_sold, 2)                         AS revenue_per_seat,
    ROUND(net_revenue / routes, 2)                             AS revenue_per_route
FROM airline_revenue
ORDER BY rank
LIMIT 10
"""))

Databricks visualization. Run in Databricks to view.

### Interpretation & Management Action

The 10-airline market is **nearly perfectly competitive**: every airline operates all 100 routes and each captures **$9.4M–$9.8M (9.8–10.3% share)**, with the leader (AmeriJet, $9.78M) only 4.3% ahead of the last-place carrier (SkyWings International, $9.38M). AmeriJet wins on volume (5,701 seats sold) while **Pacific Air wins on yield ($1,716/seat on the fewest seats)** — the only meaningful spread is $45/seat between the best and worst yields, confirming that neither network nor pricing confers a durable advantage.

**Suggested management action:** With shares this flat, market share can only move through volume tactics — schedule convenience, distribution reach, and loyalty lock-in — rather than price wars that flat yields can't fund; benchmark Pacific Air's premium-yield playbook, and evaluate consolidation or alliance partnerships, since scale advantages are currently absent across the entire carrier field.

## 11. Revenue Share of the Top 10 Airlines

Combined revenue share of the 10 highest-revenue airlines. Note: the network operates exactly 10 airlines, so the top 10 is the entire market — the query therefore also reports the largest/smallest carrier shares and the Herfindahl–Hirschman Index (HHI) to characterize concentration.

In [0]:
display(spark.sql(f"""
WITH airline_revenue AS (
    SELECT
        a.airline_id,
        SUM(fb_ticketprice * (1 - fb_discount) * fb_seats) AS net_revenue
    FROM {catalog}.silver.flight_booking fb
    JOIN {catalog}.gold.dim_airline a
      ON fb.fb_airlineid = a.airline_id
    WHERE fb_refundflag <> 'R'          -- exclude refunded segments
    GROUP BY a.airline_id
),
ranked AS (
    SELECT
        net_revenue,
        ROW_NUMBER() OVER (ORDER BY net_revenue DESC) AS rank,
        COUNT(*) OVER ()                              AS total_airlines,
        SUM(net_revenue) OVER ()                      AS total_revenue
    FROM airline_revenue
)
SELECT
    total_airlines,
    ROUND(total_revenue, 2)                                     AS total_revenue,
    ROUND(SUM(CASE WHEN rank <= 10 THEN net_revenue END), 2)     AS top_10_revenue,
    ROUND(100 * SUM(CASE WHEN rank <= 10 THEN net_revenue END)
          / total_revenue, 2)                                    AS top_10_share_pct,
    ROUND(MAX(net_revenue), 2)                                  AS largest_airline_revenue,
    ROUND(100 * MAX(net_revenue) / total_revenue, 2)             AS largest_share_pct,
    ROUND(MIN(net_revenue), 2)                                  AS smallest_airline_revenue,
    ROUND(100 * MIN(net_revenue) / total_revenue, 2)             AS smallest_share_pct,
    ROUND(SUM(POWER(100 * net_revenue / total_revenue, 2)), 0)   AS hhi_index
FROM ranked
GROUP BY total_airlines, total_revenue
"""))

### Interpretation & Management Action

The top 10 airlines account for **100% of the $95.25M market net revenue — because the network operates exactly 10 carriers**, so the ranking covers the entire competitive field. Concentration is minimal: the largest carrier (AmeriJet) holds just **10.26%** and the smallest (SkyWings International) **9.84%**, a spread of only 0.4 percentage points, and the **HHI of ~1,000** sits well below the 1,500 threshold that US antitrust guidance uses to define an unconcentrated market — statistically this is as close to a perfectly competitive market as 10 firms can get.

**Suggested management action:** No carrier has pricing power or a share moat, so competitive strategy must run through differentiation — loyalty programs, schedule quality, and corporate contracts — rather than price cutting; regulators would view any merger among these carriers as low-risk for concentration, so inorganic growth via consolidation is a viable strategic option if unit economics justify it.

## 12. Revenue by Passenger Nation (All Nations)

Net revenue, bookings, and passengers by passenger nationality, joining the booking base to the gold dimensions `sl.gold.dim_passenger` and `sl.gold.dim_geography`. All nations are listed, ranked by revenue.

In [0]:
display(spark.sql(f"""
WITH nation_revenue AS (
    SELECT
        g.nation_name,
        g.region_name,
        COUNT(DISTINCT b.bk_bookingid)   AS bookings,
        COUNT(DISTINCT b.bk_passengerid) AS passengers,
        SUM(b.bk_totalprice)             AS revenue
    FROM {catalog}.silver.booking b
    JOIN {catalog}.gold.dim_passenger p
      ON b.bk_passengerid = p.passenger_id
    JOIN {catalog}.gold.dim_geography g
      ON p.nation_key = g.nation_key
    GROUP BY g.nation_name, g.region_name
)
SELECT
    nation_name,
    region_name,
    passengers,
    bookings,
    ROUND(revenue, 2)                              AS net_revenue,
    ROUND(100 * revenue / SUM(revenue) OVER (), 2)  AS revenue_share_pct,
    ROUND(revenue / passengers, 2)                 AS revenue_per_passenger,
    ROUND(revenue / bookings, 2)                   AS avg_booking_value
FROM nation_revenue
ORDER BY net_revenue DESC
"""))

Databricks visualization. Run in Databricks to view.

### Interpretation & Management Action

All **25 nations contribute meaningfully** — shares range only **2.84% (Colombia) to 4.86% (Algeria)** — so no single nationality is a dependency, echoing the balanced-concentration pattern seen across routes, airlines, and travel purposes. The leaders are **Algeria ($2.94M), France ($2.89M), and Turkey ($2.83M)**, and per-passenger value varies meaningfully: **Turkish flyers are worth $46.4K each** versus $37.3K for UK and $37.1K for Egyptian passengers — a 25% spread that locale-level economics do create, even where aggregate shares look flat.

**Suggested management action:** Allocate growth marketing to the under-penetrated tail (Colombia, Egypt, Mexico — the three lowest shares) with localized acquisition campaigns, protect the high-value markets (Turkey, UAE, Algeria) with premium service and loyalty investment, and set per-nation revenue targets in the 3–5% share band rather than chasing a single flagship market.

## 13. Revenue and Volume Split by Refund Flag

Segments, seats, bookings, and revenue by `fb_refundflag` (N = no refund, A = refunded-applied, R = refunded) from the flight_booking fact behind the route/airline gold analyses. Gross revenue = ticket price × (1 − discount) × seats; net revenue sets refunded segments to zero.

In [0]:
display(spark.sql(f"""
WITH flag_split AS (
    SELECT
        fb_refundflag                                   AS refund_flag,
        COUNT(*)                                        AS segments,
        SUM(fb_seats)                                   AS seats,
        COUNT(DISTINCT fb_bookingid)                    AS bookings,
        SUM(fb_ticketprice * (1 - fb_discount) * fb_seats) AS gross_revenue
    FROM {catalog}.silver.flight_booking
    GROUP BY fb_refundflag
)
SELECT
    refund_flag,
    segments,
    ROUND(100 * segments / SUM(segments) OVER (), 1)              AS segment_share_pct,
    seats,
    ROUND(100 * seats / SUM(seats) OVER (), 1)                     AS seat_share_pct,
    bookings,
    ROUND(gross_revenue, 2)                                        AS gross_revenue,
    ROUND(100 * gross_revenue / SUM(gross_revenue) OVER (), 1)      AS gross_revenue_share_pct,
    ROUND(CASE WHEN refund_flag = 'R' THEN 0
                ELSE gross_revenue END, 2)                         AS net_revenue
FROM flag_split
ORDER BY gross_revenue DESC
"""))

Databricks visualization. Run in Databricks to view.

### Interpretation & Management Action

Refund leakage is material: **10% of segments (2,504) carrying 6,339 seats were fully refunded, wiping out $10.65M — 10.1% of the $105.9M gross revenue**. The non-refunded base (flag N) carries 65% of volume and revenue, flag A another ~25%, and the volume and revenue splits track each other almost exactly (65/25/10 on both measures) — meaning refund propensity is uniform across price points, not concentrated in cheap or premium fares. One caveat: the exact semantics of flag A (likely an applied/partial refund state) should be confirmed with the data steward, since only R is treated as zero-revenue here.

**Suggested management action:** Attack the $10.65M refund leakage — tighten refund eligibility, price refundability into fares (non-refundable discounts), and analyze which routes, cabins, and booking lead-times produce the 2,153 refunded bookings so policy targets the actual drivers; and confirm the A flag's meaning so net-revenue reporting isn't overstating recoverable revenue.

## 14. Average Flight Duration (Departure to Arrival, in Days) by Cabin Class

Average, minimum, and maximum departure-to-arrival day difference by cabin class, from the flight_booking fact behind the route/airline gold analyses.

In [0]:
display(spark.sql(f"""
WITH cabin_duration AS (
    SELECT
        fb_cabinclass                                              AS cabin_class,
        COUNT(*)                                                  AS segments,
        SUM(fb_seats)                                             AS seats,
        AVG(DATEDIFF(fb_arrivaldate, fb_departuredate))            AS avg_flight_days,
        MIN(DATEDIFF(fb_arrivaldate, fb_departuredate))             AS min_flight_days,
        MAX(DATEDIFF(fb_arrivaldate, fb_departuredate))             AS max_flight_days
    FROM {catalog}.silver.flight_booking
    GROUP BY fb_cabinclass
)
SELECT
    cabin_class,
    segments,
    ROUND(100 * segments / SUM(segments) OVER (), 1)   AS segment_share_pct,
    seats,
    ROUND(avg_flight_days, 2)                          AS avg_flight_duration_days,
    min_flight_days,
    max_flight_days
FROM cabin_duration
ORDER BY segments DESC
"""))

### Interpretation & Management Action

**Average flight duration is 0.0 days in every cabin class** — all 25,000 segments have arrival date equal to departure date (min = max = 0), so at the day level every flight is same-day regardless of cabin. This is a data-granularity limit, not a business finding: the fact stores only date-typed departure/arrival fields with no times, so sub-day block time (the real differentiator between a short-haul economy hop and an ultra-long-haul first-class flight) cannot be measured. What the data does confirm is the cabin mix: **Economy carries 60.5% of segments, Premium Economy 19.8%, Business 14.8%, and First just 5.0%**.

**Suggested management action:** Instrument the booking pipeline to capture timestamp-level (or scheduled block-time) departure and arrival values so duration-based cabin analytics become possible — premium-cabin performance on long flights is a classic yield lever that this dataset currently cannot see; until then, use route_type (from Query 9) as the proxy for trip length when analyzing cabin performance.

## 15. Booking Count and Revenue by Booking Priority

Bookings and revenue by `bk_bookingpriority` (1-URGENT through 5-STANDARD), from the booking base behind the gold KPI view `sl.gold.vw_annual_booking_kpi`.

In [0]:
display(spark.sql(f"""
WITH priority_split AS (
    SELECT
        bk_bookingpriority             AS booking_priority,
        COUNT(DISTINCT bk_bookingid)   AS bookings,
        SUM(bk_totalprice)            AS revenue
    FROM {catalog}.silver.booking
    GROUP BY bk_bookingpriority
)
SELECT
    booking_priority,
    bookings,
    ROUND(100 * bookings / SUM(bookings) OVER (), 1)  AS booking_share_pct,
    ROUND(revenue, 2)                                AS net_revenue,
    ROUND(100 * revenue / SUM(revenue) OVER (), 1)     AS revenue_share_pct,
    ROUND(revenue / bookings, 2)                      AS avg_booking_value
FROM priority_split
ORDER BY booking_priority
"""))

Databricks visualization. Run in Databricks to view.

### Interpretation & Management Action

Booking priority carries **no revenue signal**: the five levels split volume almost perfectly evenly (19.4–21.2% of bookings, $11.79M–$12.64M revenue each), and average booking values cluster tightly at $7.5K–$7.6K. Curiously, **2-HIGH leads on both bookings (1,695) and revenue ($12.64M) while 1-URGENT ranks second-lowest** (1,571 bookings, $11.86M) — the most urgent bookings are not the most valuable, so urgency and commercial value are unrelated in this data.

**Suggested management action:** Audit how booking priority is assigned — if it's meant to triage ops resources, the field currently misdirects attention (urgent ≠ valuable); re-anchor priority on commercial and service criteria (booking value, passenger tier, travel purpose) so urgent handling maps to revenue at risk, and standardize the assignment rules in the booking pipeline.

## 16. Monthly Breakdown of Booking Status

Bookings per month by status (`bk_status`: F = final, O = open, C = cancelled) with status shares, from the booking base behind the gold KPI view `sl.gold.vw_annual_booking_kpi`, across all years (1994–1998).

In [0]:
display(spark.sql(f"""
WITH monthly AS (
    SELECT
        DATE_TRUNC('month', bk_bookingdate)                                 AS month,
        COUNT(*)                                                        AS total_bookings,
        SUM(CASE WHEN bk_status = 'F' THEN 1 ELSE 0 END)                 AS final_bookings,
        SUM(CASE WHEN bk_status = 'O' THEN 1 ELSE 0 END)                 AS open_bookings,
        SUM(CASE WHEN bk_status = 'C' THEN 1 ELSE 0 END)                 AS cancelled_bookings
    FROM {catalog}.silver.booking
    WHERE bk_bookingdate IS NOT NULL
    GROUP BY DATE_TRUNC('month', bk_bookingdate)
)
SELECT
    month,
    year(month)                AS year,
    date_format(month, 'MMMM') AS month_name,
    total_bookings,
    final_bookings,
    ROUND(100 * final_bookings / total_bookings, 1)      AS final_share_pct,
    open_bookings,
    ROUND(100 * open_bookings / total_bookings, 1)        AS open_share_pct,
    cancelled_bookings,
    ROUND(100 * cancelled_bookings / total_bookings, 1)   AS cancelled_share_pct
FROM monthly
ORDER BY month
"""))

Databricks visualization. Run in Databricks to view.

### Interpretation & Management Action

The status mix is stable and healthy across all 60 months: roughly **75% of bookings reach final status, ~14% stay open, and ~10% cancel**, with no deteriorating trend over the 5 years. Two patterns stand out — an **open-bookings backlog in Aug-Oct 1996** (open share peaking at **24.5% in Aug 1996** vs the ~14% norm, suggesting a payment/processing bottleneck), and a recurring **September cancellation uptick** (13.1-14.2% in 1994, 1995, and 1998 vs ~10% baseline); the business exits at its healthiest, with **Dec 1998 at 82.8% final and just 7.3% cancelled**.

**Suggested management action:** Root-cause the Aug-Oct 1996 open-booking backlog to harden the booking/payment pipeline against a repeat, add ops alerts when monthly open or cancelled share breaches ~15% / ~13%, and study the September cancellation spike (post-summer schedule changes?) for a targeted rebooking/retention offer each September.

## 17. Route Revenue Ranking Within Each Travel Purpose

Top 3 routes by net revenue (ticket price × (1 − discount) × seats, refunded segments excluded) within each of the five travel purposes, joining the flight_booking fact through the booking base to the gold dimensions `sl.gold.dim_passenger` and `sl.gold.dim_route`.

In [0]:
display(spark.sql(f"""
WITH route_purpose_revenue AS (
    SELECT
        p.travel_purpose,
        r.route_id,
        r.route_name,
        r.route_type,
        COUNT(DISTINCT fb.fb_bookingid)                    AS bookings,
        SUM(fb_ticketprice * (1 - fb_discount) * fb_seats) AS net_revenue
    FROM {catalog}.silver.flight_booking fb
    JOIN {catalog}.silver.booking b
      ON fb.fb_bookingid = b.bk_bookingid
    JOIN {catalog}.gold.dim_passenger p
      ON b.bk_passengerid = p.passenger_id
    JOIN {catalog}.gold.dim_route r
      ON fb.fb_routeid = r.route_id
    WHERE fb_refundflag <> 'R'          -- exclude refunded segments
    GROUP BY p.travel_purpose, r.route_id, r.route_name, r.route_type
),
ranked AS (
    SELECT
        travel_purpose,
        route_id,
        route_name,
        route_type,
        bookings,
        net_revenue,
        ROW_NUMBER() OVER (PARTITION BY travel_purpose
                           ORDER BY net_revenue DESC)                    AS rank_in_purpose,
        ROUND(100 * net_revenue
              / SUM(net_revenue) OVER (PARTITION BY travel_purpose), 2) AS share_within_purpose_pct
    FROM route_purpose_revenue
)
SELECT
    travel_purpose,
    rank_in_purpose,
    route_id,
    route_name,
    route_type,
    bookings,
    ROUND(net_revenue, 2)   AS net_revenue,
    share_within_purpose_pct
FROM ranked
WHERE rank_in_purpose <= 3
ORDER BY travel_purpose, rank_in_purpose
"""))

Databricks visualization. Run in Databricks to view.

### Interpretation & Management Action

Route revenue is flat within every travel purpose: even the **#1 route holds only 1.4-1.65% of its purpose's revenue** (tops of $260K-$310K each), so no purpose depends on a flagship route - mirroring the portfolio-wide balance seen in Queries 7-11. Three routes emerge as **cross-purpose backbones - LOS-MIA (top-3 in BUSINESS, MILITARY, STUDENT), GRU-MAD (CREW, LEISURE), and SIN-LIM (BUSINESS, MILITARY)** - while each purpose also has a distinct leader: BUSINESS tops out on ultra-long-haul SIN-MAD, LEISURE on domestic PEK-MNL, MILITARY on DXB-DEL, and STUDENT on MEX-DEL.

**Suggested management action:** Treat LOS-MIA, GRU-MAD, and SIN-LIM as backbone assets - prioritize their schedule reliability, capacity, and service recovery since disruption there hits multiple traveler segments at once; run purpose-tailored campaigns on each segment's distinct leader (corporate contracts around SIN-MAD, leisure bundles on PEK-MNL); and set per-purpose route targets in the 1-2% share band, matching the naturally flat distribution.

## 18. Revenue Comparison: Passenger Geography vs Airline Geography by Region

The same line-item net revenue pool (ticket price times (1 - discount) times seats, refunded segments excluded) attributed two ways: by the passenger home region (via booking, gold dim_passenger, gold dim_geography) and by the operating airline headquarters region (via gold dim_airline, gold dim_geography). Shares and gaps are compared per region.

In [0]:
display(spark.sql(f"""
WITH passenger_geo AS (
    SELECT
        gp.region_name                                     AS region,
        SUM(fb_ticketprice * (1 - fb_discount) * fb_seats) AS passenger_geo_revenue
    FROM {catalog}.silver.flight_booking fb
    JOIN {catalog}.silver.booking b
      ON fb.fb_bookingid = b.bk_bookingid
    JOIN {catalog}.gold.dim_passenger p
      ON b.bk_passengerid = p.passenger_id
    JOIN {catalog}.gold.dim_geography gp
      ON p.nation_key = gp.nation_key
    WHERE fb_refundflag <> 'R'
    GROUP BY gp.region_name
),
airline_geo AS (
    SELECT
        ga.region_name                                     AS region,
        SUM(fb_ticketprice * (1 - fb_discount) * fb_seats) AS airline_geo_revenue
    FROM {catalog}.silver.flight_booking fb
    JOIN {catalog}.gold.dim_airline a
      ON fb.fb_airlineid = a.airline_id
    JOIN {catalog}.gold.dim_geography ga
      ON a.nation_key = ga.nation_key
    WHERE fb_refundflag <> 'R'
    GROUP BY ga.region_name
),
combined AS (
    SELECT
        COALESCE(p.region, a.region) AS region,
        p.passenger_geo_revenue,
        a.airline_geo_revenue
    FROM passenger_geo p
    FULL OUTER JOIN airline_geo a
      ON p.region = a.region
)
SELECT
    region,
    ROUND(passenger_geo_revenue, 2)                          AS passenger_origin_revenue,
    ROUND(100 * passenger_geo_revenue
          / SUM(passenger_geo_revenue) OVER (), 2)             AS passenger_origin_share_pct,
    ROUND(airline_geo_revenue, 2)                             AS airline_hq_revenue,
    ROUND(100 * airline_geo_revenue
          / SUM(airline_geo_revenue) OVER (), 2)               AS airline_hq_share_pct,
    ROUND(passenger_geo_revenue - airline_geo_revenue, 2)     AS revenue_difference,
    ROUND(100 * passenger_geo_revenue / SUM(passenger_geo_revenue) OVER ()
        - 100 * airline_geo_revenue / SUM(airline_geo_revenue) OVER (), 2) AS share_gap_pct_points
FROM combined
ORDER BY region
"""))

Databricks visualization. Run in Databricks to view.

### Interpretation & Management Action

The network is geographically balanced on both measures: every region contributes **18.5-21.7% of the $95.25M revenue pool** whether attributed by passenger origin or airline headquarters, and no share gap exceeds ~1.8 points. The two imbalances are meaningful though: **EUROPE is a net revenue importer (+1.82 pts, +$1.73M)** - European-origin passengers outspend what Europe-based carriers (SkyWings, Nordic Air) capture - while **AMERICA is a net exporter (-1.75 pts, -$1.67M)** - US-headquartered carriers (AmeriJet, Trans-Atlantic Airways) earn more than American-origin passengers generate, meaning they successfully carry foreign traffic.

**Suggested management action:** Treat the gaps as market-positioning signals, not problems: replicate the US carriers' cross-border capture playbook (interline partnerships, foreign-market distribution) for the other regions' flag carriers, and have European carriers defend their home-origin market where they under-index ~2 points; keep the naturally balanced network intact since no region is structurally over- or under-exposed.